In [1]:
!pip install --upgrade torch transformers

In [2]:
!pip install datasets transformers accelerate

In [3]:
#load the dataset
from datasets import load_dataset

# Load your CSV dataset
data = load_dataset("csv", data_files="dataset/193k.csv")
data = data['train'].select(range(0, 1000))  # Select the first 1000 rows (you can adjust this number)


data


Dataset({
    features: ['title', 'text'],
    num_rows: 1000
})

In [4]:
#load gpt2 tokenizer
from transformers import GPT2Tokenizer

# Load the GPT-2 tokenizer
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token  # GPT-2 uses the end of sequence token as padding


In [5]:
def tokenize_function(examples):
    # Tokenize title and text for each example in the batch
    titles = tokenizer(examples['title'], truncation=True, padding='max_length', max_length=512)
    texts = tokenizer(examples['text'], truncation=True, padding='max_length', max_length=512)

    # Combine tokenized input_ids and attention_mask for title and text
    input_ids = [title_ids + text_ids for title_ids, text_ids in zip(titles['input_ids'], texts['input_ids'])]
    attention_mask = [title_mask + text_mask for title_mask, text_mask in zip(titles['attention_mask'], texts['attention_mask'])]

    # Create labels that ignore the title part by filling with -100
    labels = [[-100] * len(title_ids) + text_ids for title_ids, text_ids in zip(titles['input_ids'], texts['input_ids'])]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

# Apply tokenization to the dataset
tokenized_datasets = data.map(tokenize_function, remove_columns=['title', 'text'], batched=True)


In [6]:
from datasets import DatasetDict

# Apply tokenization to the smaller dataset
tokenized_datasets = data.map(tokenize_function, remove_columns=['title', 'text'], batched=True)

# Shuffle the dataset (if not already shuffled)
tokenized_datasets = tokenized_datasets.shuffle(seed=42)

# Split into training and validation sets (80% train, 20% validation)
train_size = int(0.8 * len(tokenized_datasets))
train_dataset = tokenized_datasets.select(range(train_size))
eval_dataset = tokenized_datasets.select(range(train_size, len(tokenized_datasets)))

# Create a DatasetDict for training and validation
datasets = DatasetDict({"train": train_dataset, "validation": eval_dataset})



In [7]:
#Define the Data Collator
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)  


In [8]:
#load the gpt2 model
from transformers import GPT2LMHeadModel

model = GPT2LMHeadModel.from_pretrained('gpt2')


In [9]:
#Training Arguments
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir='./results',           # Directory to save model checkpoints
    overwrite_output_dir=True,        # Overwrite previous checkpoints
    num_train_epochs=3,               # Number of training epochs
    per_device_train_batch_size=4,    # Batch size for training
    per_device_eval_batch_size=4,     # Batch size for evaluation
    warmup_steps=500,                 # Warmup steps for learning rate scheduler
    weight_decay=0.01,                # Weight decay for optimizer
    logging_dir='./logs',             # Directory for logs
    logging_steps=250,                 # Log every 250 steps
    eval_strategy ="epoch",      # Evaluate at the end of every epoch
    save_strategy="epoch"             # Save checkpoint at the end of every epoch
)


In [10]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator
)


In [11]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,No log,3.246823
2,3.770400,3.222121
3,3.241800,3.230494


TrainOutput(global_step=600, training_loss=3.4442481486002605, metrics={'train_runtime': 24565.497, 'train_samples_per_second': 0.098, 'train_steps_per_second': 0.024, 'total_flos': 1254201753600000.0, 'train_loss': 3.4442481486002605, 'epoch': 3.0})

In [12]:
trainer.evaluate()


{'eval_loss': 3.230494499206543,
 'eval_runtime': 337.8835,
 'eval_samples_per_second': 0.592,
 'eval_steps_per_second': 0.148,
 'epoch': 3.0}

In [13]:
# Ensure the model is set to evaluation mode.
model.eval()


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2SdpaAttention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [16]:
def generate_text(prompt, max_length=300):
    # Tokenize the input prompt with padding and attention mask
    inputs = tokenizer(prompt, return_tensors="pt", padding=True, truncation=True)
    
    # Generate the output text
    output = model.generate(
        inputs['input_ids'], 
        attention_mask=inputs['attention_mask'],  # Set attention mask
        max_length=max_length,  # Maximum length for the generated text
        num_return_sequences=1,  # Number of sequences to return
        no_repeat_ngram_size=2,  # Prevent repeating bigrams
        early_stopping=False,     # Stop early if a natural stopping point is found
        pad_token_id=tokenizer.eos_token_id  # Set the padding token ID
    )
    
    # Decode the generated text to a string
    generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
    
    return generated_text



In [17]:
prompt = "Describe the basic steps of machine Learning?"
generated_output = generate_text(prompt, max_length=300)
print(generated_output)


Describe the basic steps of machine Learning?

Machine Learning is a new field that is gaining popularity in the last few years. It is the most popular field in computer science and is used to solve complex problems. Machine Learning has been used for many years to develop algorithms for predicting the future.
“Machine learning is an important part of the science of artificial intelligence.”
. The term refers to the process of learning and training algorithms to improve the performance of a machine learning algorithm. In this article, we will discuss the basics of Machine learning. We will use the term “machine learning’ to describe the fundamental steps in machine training. This article will focus on the fundamentals of this field. For more information, please refer to our article on MachineLearning.com.


The basic principles of training a neural network
A neural net is one of two types of neural networks. One is called a ‘neural network‘ and the other is referred to as a training ne

In [18]:
#saving the model
model.save_pretrained('results/gpt2_finetuned')
tokenizer.save_pretrained('results/gpt2_finetuned')


('results/gpt2_finetuned/tokenizer_config.json',
 'results/gpt2_finetuned/special_tokens_map.json',
 'results/gpt2_finetuned/vocab.json',
 'results/gpt2_finetuned/merges.txt',
 'results/gpt2_finetuned/added_tokens.json')

In [ ]:
#for next time 
from transformers import GPT2LMHeadModel, GPT2Tokenizer

# Load the fine-tuned GPT-2 model and tokenizer
model = GPT2LMHeadModel.from_pretrained('results/gpt2_finetuned')
tokenizer = GPT2Tokenizer.from_pretrained(results/gpt2_finetuned')

# Set the model to evaluation mode
model.eval()
